# M3 — Modelado

Entrena y compara los cinco modelos (naïve, Regresión Logística, Random
Forest, XGBoost, SVM) con validación temporal `TimeSeriesSplit` y test
*out-of-sample* de 2025.

Corresponde a la Sección *Módulo 3* del Capítulo 4 (OE3).

In [ ]:
import pandas as pd
from tfm import data, features, metrics, validation
from tfm.models import build_model_specs

raw = data.load_raw()
ticker = "AAPL"  # change or loop over config.TICKERS

dataset = features.build_dataset(raw[ticker])
train, test = validation.temporal_split(dataset)
cols = features.feature_columns(dataset)
X_train, y_train = train[cols], train["target"]
X_test, y_test = test[cols], test["target"]

In [ ]:
results, fitted = [], {}
for name, spec in build_model_specs().items():
    model, best = validation.tune_model(spec, X_train, y_train)
    fitted[name] = model
    row = metrics.classification_metrics(y_test, model.predict(X_test))
    results.append({"model": name, **row, "best_params": best})
pd.DataFrame(results).set_index("model")